Textrank for document summarization


In [ ]:
  #importing basic libraries
import numpy as np
import nltk
import networkx as nx
from nltk.tokenize import sent_tokenize,word_tokenize
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

In [ ]:
#download nltk resources
nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
def preprocess_text(text):
  stop_words = set(stopwords.words('english'))
  sentences=sent_tokenize(text)
  word_frequencies = []
  for sent in sentences:
    words = word_tokenize(sent.lower())
    filtered_words = [w for w in words if w.isalnum() and w not in stop_words]
    word_frequencies=Counter(filtered_words)
  return sentences,word_frequencies

In [ ]:
def similarity_matrix(word_frequencies):
  size=len(word_frequencies)
  sm=np.zeros((size,size))
  for i in range(size):
    for j in range(size):
      if i!=j:
        words1=word_frequencies[i]
        words2=word_frequencies[j]
        common_words=set(words1.keys()).union(set(words2.keys()))

        vec1 = np.array([words1[word] for word in common_words])
        vec2 = np.array([words2[word] for word in common_words])

        sm[i][j]=cosine_similarity([vec1],[vec2])[0,0]
      return sm

In [ ]:
def textrank(text,num=3):
  sentences,word_frequencies=preprocess_text(text)
  sm=similarity_matrix(word_frequencies)

  #build graph and apply pagerank

  graph=nx.from_numpy_array(sm)
  scores=nx.pagerank(graph)

  ranked_sentences=sorted(((scores.get(i,0),s)
            for i,s in enumerate(sentences)),reverse=True)
  summary=" ".join(sent for _,sent in ranked_sentences[:num])
  return summary

In [ ]:
text = """India, officially the Republic of India,[j][20] is a country in South Asia. It is the seventh-largest country by area; the most populous country since 2023;[21] and, since its independence in 1947, the world's most populous democracy.[22][23][24] Bounded by the Indian Ocean on the south, the Arabian Sea on the southwest, and the Bay of Bengal on the southeast, it shares land borders with Pakistan to the west;[k] China, Nepal, and Bhutan to the north; and Bangladesh and Myanmar to the east. In the Indian Ocean, India is near Sri Lanka and the Maldives; its Andaman and Nicobar Islands share a maritime border with Myanmar, Thailand, and Indonesia.

Modern humans arrived on the Indian subcontinent from Africa no later than 55,000 years ago.[26][27][28] Their long occupation, predominantly in isolation as hunter-gatherers, has made the region highly diverse.[29] Settled life emerged on the subcontinent in the western margins of the Indus river basin 9,000 years ago, evolving gradually into the Indus Valley Civilisation of the third millennium BCE.[30] By 1200 BCE, an archaic form of Sanskrit, an Indo-European language, had diffused into India from the northwest.[31][32] Its hymns recorded the early dawnings of Hinduism in India.[33] India's pre-existing Dravidian languages were supplanted in the northern regions.[34] By 400 BCE, caste had emerged within Hinduism,[35] and Buddhism and Jainism had arisen, proclaiming social orders unlinked to heredity.[36] Early political consolidations gave rise to the loose-knit Maurya and Gupta Empires.[37] This era was noted for creativity in art, architecture, and writing,[38] but the status of women declined,[39] and untouchability became an organised belief.[l][40] In South India, the Middle kingdoms exported Dravidian language scripts and religious cultures to the kingdoms of Southeast Asia.[41]"""

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
print(textrank(text,num=5))

India, officially the Republic of India,[j][20] is a country in South Asia. [l][40] In South India, the Middle kingdoms exported Dravidian language scripts and religious cultures to the kingdoms of Southeast Asia. [41] [37] This era was noted for creativity in art, architecture, and writing,[38] but the status of women declined,[39] and untouchability became an organised belief. [36] Early political consolidations gave rise to the loose-knit Maurya and Gupta Empires.
